# 17.3 Other piecewise-linear functions

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Many simple piecewise-linear functions can be modeled in several equivalent ways in
AMPL. The function of [Figure 17-4](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-4), for example, could be written as

```ampl
if Use[t] > avail_min[t]
       then time_penalty[t] * (Use[t] - avail_min[t]) else 0
```

or more concisely as

```ampl
max(0, time_penalty[t] * (Use[t] - avail_min[t]))
```

The current version of AMPL does not detect that these expressions are piecewise-linear,
so you are unlikely to get satisfactory results if you try to solve a model that has expressions
like these in its objective. To take advantage of linear programming techniques that
can be applied for piecewise-linear terms, you need to use the piecewise-linear terminology

```ampl
<<avail_min[t]; 0,time_penalty[t]>> Use[t]
```

so the structure can be noted and passed to a solver.

The same advice applies to the function abs. Imagine that we would like to encourage
the number of hours used to be close to avail_min[t]. Then we would want the
penalty term to equal time_penalty[t] times the amount that Use[t] deviates
from avail_min[t], either above or below. Such a term can be written as

```ampl
time_penalty[t] * abs(Use[t] - avail_min[t])
```

To express it in an explicitly piecewise-linear fashion, however, you should write it as

```ampl
time_penalty[t] * <<avail_min[t]; -1,1>> Use[t]
```

or equivalently,

```ampl
<<avail_min[t]; -time_penalty[t],time_penalty[t]>> Use[t]
```

As this example shows, multiplying a piecewise-linear function by a constant is the same
as multiplying each of its slopes individually.

As a final example of a common piecewise-linearity in the objective, we return to the
kind of assignment model that was discussed in [Chapter 15](../15/15.md). Recall that, for `i` in the set
PEOPLE and `j` in the set PROJECTS, cost[i,j] is the cost for person `i` to work an
hour on project j, and the decision variable Assign[i,j] is the number of hours that
person `i` is assigned to work on project j:

```ampl
set PEOPLE;
set PROJECTS;
param cost {PEOPLE,PROJECTS} >= 0;
var Assign {PEOPLE,PROJECTS} >= 0;
```

We originally formulated the objective as the total cost of all assignments,

```ampl
sum {i in PEOPLE, j in PROJECTS} cost[i,j] * Assign[i,j]
```

What if we want the fairest assignment instead of the cheapest? Then we might minimize
the maximum cost of any one person's assignments:

```ampl
set PEOPLE;
set PROJECTS;
param supply {PEOPLE} >= 0;   # hours each person is available
param demand {PROJECTS} >= 0; # hours each project requires
              check: sum {i in PEOPLE} supply[i]
                     = sum {j in PROJECTS} demand[j];
param cost {PEOPLE,PROJECTS} >= 0;    # cost per hour of work
param limit {PEOPLE,PROJECTS} >= 0;   # maximum contributions
                                          # to projects
var M;
var Assign {i in PEOPLE, j in PROJECTS} >= 0, <= limit[i,j];
minimize Max_Cost: M;
subject to M_def {i in PEOPLE}:
       M >= sum {j in PROJECTS} cost[i,j] * Assign[i,j];
subject to Supply {i in PEOPLE}:
       sum {j in PROJECTS} Assign[i,j] = supply[i];
subject to Demand {j in PROJECTS}:
       sum {i in PEOPLE} Assign[i,j] = demand[j];
```

**[Figure 17-9](./17_3_other_piecewiselinear_functions.ipynb#fig-17-9):** Min-max assignment model (minmax.mod).

```ampl
minimize Max_Cost:
       max {i in PEOPLE}
       sum {j in PROJECTS} cost[i,j] * Assign[i,j];
```

This function is also piecewise-linear, in a sense; it is pieced together from the linear
functions sum {j in PROJECTS} cost[i,j] * Assign[i,j] for different people
i. However, it is not piecewise-linear in the individual variables — in mathematical jargon,
it is not separable — and hence it cannot be written using the << . . . >> notation.

This is a case in which piecewise-linearity can only be handled by rewriting the model
as a linear program. We introduce a new variable M to represent the maximum. Then we
write constraints to guarantee that M is greater than or equal to each cost of which it is the
maximum:

```ampl
var M;
minimize Max_Cost: M;
subject to M_def {i in PEOPLE}:
       M >= sum {j in PROJECTS} cost[i,j] * Assign[i,j];
```

Because M is being minimized, at the optimal solution it will in fact equal the maximum
of sum {j in PROJECTS} cost[i,j] * Assign[i,j] over all `i` in PEOPLE.
The other constraints are the same as in any assignment problem, as shown in [Figure 17-9](./17_3_other_piecewiselinear_functions.ipynb#fig-17-9).
This kind of reformulation can be applied to any problem that has a "min-max"
objective. The same idea works for the analogous "max-min" objective, with
maximize instead of minimize and with M <= . . . in the constraints.